<a href="https://colab.research.google.com/github/padhisneha2025-dev/SIH-Prototype/blob/main/SIH_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

csv_content = """Location_ID,Location_Name,Type,Latitude,Longitude,Crop,Available_Kg,Modal_Price_Per_Quintal,Distance_To_Hub_Km,APMC_Cess_Percent,Arrival_Tons,Farmer_Name,Quality_Grade,Harvest_Date
M01,Brahmapur APMC,Mandi,19.3150,84.7941,Tomato,0,2200,0.0,1.5,45,N/A,N/A,N/A
M02,Hinjilicut Mandi,Mandi,19.4817,84.7449,Tomato,0,2450,19.2,1.5,30,N/A,N/A,N/A
M03,Digapahandi Market,Mandi,19.3700,84.5800,Tomato,0,2650,23.3,2.0,22,N/A,N/A,N/A
F01,Kukudakhandi,Farmer_Lot,19.3926,84.7521,Tomato,450,0,9.7,0,0,Ramesh Prusty,Grade A,2026-08-20
F02,Nimakhandi,Farmer_Lot,19.4230,84.7941,Tomato,320,0,12.0,0,0,Suresh Naik,Grade B,2026-08-21
F03,Kanchuru,Farmer_Lot,19.3150,84.9080,Tomato,600,0,12.0,0,0,Bijay Behera,Grade A,2026-08-19
F04,Ambapua,Farmer_Lot,19.2380,84.7130,Tomato,380,0,12.0,0,0,Manoj Sahoo,Grade B,2026-08-20
F05,Lochapada,Farmer_Lot,19.1890,84.7941,Tomato,500,0,14.0,0,0,Krishna Panda,Grade A,2026-08-22"""

# Saving this string directly to a CSV file
with open("sih_market_data.csv", "w") as file:
    file.write(csv_content)

print("Success: sih_market_data.csv created!")

Success: sih_market_data.csv created!


In [4]:
def calculate_best_buyer_real_data(harvest_kg, transport_rate_per_km):
    # 1. Load the real CSV
    df = pd.read_csv("sih_market_data.csv")

    # 2. FILTERING: Keep only the Mandis! (Throw away the Farmer_Lot rows)
    df = df[df['Type'] == 'Mandi'].copy()

    # 3. Handle the Quintal Math (Convert price per 100kg to price per 1kg)
    df['price_per_kg'] = df['Modal_Price_Per_Quintal'] / 100

    # 4. Calculate Revenue
    df['revenue'] = df['price_per_kg'] * harvest_kg

    # 5. Calculate Logistics Cost
    df['transport_cost'] = df['Distance_To_Hub_Km'] * transport_rate_per_km

    # 6. Calculate the APMC Tax (Cess % of Revenue)
    df['tax_cost'] = df['revenue'] * (df['APMC_Cess_Percent'] / 100)

    # 7. Final NRV: Revenue minus Transport minus Taxes
    df['nrv'] = df['revenue'] - df['transport_cost'] - df['tax_cost']

    # 8. Sort and return winner
    df = df.sort_values(by='nrv', ascending=False)

    # We will return a few specific columns so it's easy to read
    return df[['Location_Name', 'revenue', 'transport_cost', 'tax_cost', 'nrv']].head(1)

# --- Let's test the machine with 100kg of tomatoes! ---
best_option = calculate_best_buyer_real_data(harvest_kg=100, transport_rate_per_km=10)
print(best_option)

        Location_Name  revenue  transport_cost  tax_cost     nrv
2  Digapahandi Market   2650.0           233.0      53.0  2364.0
